# Classification Multi-Label de Pathologies Oculaires
## Rétinopathie Diabétique · Glaucome · DMLA

---

**Stage Temeoo | Master Management de l'Intelligence Artificielle en Santé**
**École Centrale de Lille**

---

### Objectif

Ce notebook implémente un pipeline de **classification multi-label** permettant de détecter
simultanément trois pathologies oculaires majeures sur des images de fond d'œil :

- **RD** — Rétinopathie Diabétique
- **Glaucome**
- **DMLA** — Dégénérescence Maculaire Liée à l'Âge

Chaque modèle produit **3 sorties sigmoid indépendantes**, une par pathologie.

### Architecture du pipeline

| Module | Étape | Description |
|:------:|-------|-------------|
| **M1** | Chargement & Fusion | 6 datasets publics fusionnés (~15 648 images) |
| **M2** | Prétraitement & Augmentation | CLAHE + Albumentations, DataLoaders PyTorch |
| **M3** | Split stratifié | 70 / 15 / 15 % par patient, anti-fuite de données |
| **M4** | Modélisation | 4 architectures, stratégie 2 phases (warm-up + fine-tuning) |
| **M5** | Évaluation | AUC par maladie, seuils optimisés, Grad-CAM |

### Datasets utilisés

| Dataset | Pathologies couvertes | Images |
|---------|-----------------------|--------|
| ODIR-5K | RD, Glaucome, DMLA | 10 000 |
| APTOS 2019 | Rétinopathie Diabétique | 3 662 |
| RFMiD | RD, DMLA | 1 920 |
| REFUGE | Glaucome | 1 200 |
| G1020 | Glaucome | 1 020 |
| ORIGA | Glaucome | ~650 |
| **Total fusionné** | | **~15 648** |

### Modèles comparés

| Architecture | Paramètres | Framework |
|--------------|-----------|-----------|
| DenseNet121 | 8 M | TensorFlow / Keras |
| EfficientNetV2S | 22 M | TensorFlow / Keras |
| EfficientNetB3 | 12 M | TensorFlow / Keras |
| RETFound (ViT-Large) | 307 M | PyTorch |

> **Environnement requis** : GPU Kaggle — *Paramètres → Accélérateur → GPU T4 x2*


---

### Sommaire

1. Installation des dépendances
2. Configuration globale & reproductibilité
3. Diagnostic — inspection de la structure des datasets
4. **M1** — Chargement & fusion des 6 datasets
5. Analyse exploratoire des données (EDA)
6. **M3** — Split stratifié Train / Val / Test
7. **M2** — Prétraitement des images & augmentation
8. **M4** — Entraînement des modèles CNN (DenseNet121 · EfficientNetV2S · EfficientNetB3)
9. **M4b** — RETFound (ViT-Large) — warm-up puis fine-tuning en 2 profondeurs
10. **M5** — Évaluation finale sur le test set & optimisation des seuils
11. Explicabilité — Grad-CAM
12. Résultats finaux — comparaison des 4 modèles × 3 pathologies
13. Vérification des fichiers et de la mémoire
14. **Conclusion & bilan**


---
## Installation des dépendances

Installation des bibliothèques complémentaires requises pour ce notebook :
- **Albumentations** : augmentation des images médicales
- **TIMM** : accès aux architectures Vision Transformer (RETFound via ViT-Large)


In [2]:
!pip install albumentations timm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 76.2 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

---
## Configuration Globale & Reproductibilité

Définition du **seed global** (SEED = 42) pour garantir la reproductibilité complète
des résultats (numpy, random, Python hashseed).

Initialisation des **chemins de sortie** Kaggle :
- `OUTPUT_PATH` → `/kaggle/working/`
- `MODELS_DIR` → `/kaggle/working/models/`
- `RESULTS_DIR` → `/kaggle/working/resultats/`


In [4]:
import os, glob, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Chemins de sortie
OUTPUT_PATH = '/kaggle/working/'
MODELS_DIR  = os.path.join(OUTPUT_PATH, 'models')
RESULTS_DIR = os.path.join(OUTPUT_PATH, 'resultats')
os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

DISEASES    = ['RD', 'Glaucome', 'DMLA']
NUM_CLASSES = len(DISEASES)
IMG_SIZE    = 224
BATCH_SIZE  = 16


print(f"Seed : {SEED}")
print(f"Diseases : {DISEASES}")
print(f"Output  : {OUTPUT_PATH}")

Seed : 42
Diseases : ['RD', 'Glaucome', 'DMLA']
Output  : /kaggle/working/


---
## Imports & Chemins des Datasets

Chargement de l'ensemble des bibliothèques et définition des **chemins Kaggle**
vers les 6 datasets montés via l'outil *Add Data*.

| Variable | Dataset |
|----------|---------|
| `ODIR_PATH` | ODIR-5K |
| `APTOS_PATH` | APTOS 2019 (Gaussian Filtered) |
| `RFMID_PATH` | RFMiD |
| `GLAUCOMA_PATH` | REFUGE + G1020 + ORIGA |


In [5]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

ODIR_PATH     = '/kaggle/input/datasets/andrewmvd/ocular-disease-recognition-odir5k'
APTOS_PATH    = '/kaggle/input/datasets/sovitrath/diabetic-retinopathy-224x224-gaussian-filtered'
RFMID_PATH    = '/kaggle/input/datasets/andrewmvd/retinal-disease-classification'
GLAUCOMA_PATH = '/kaggle/input/datasets/arnavjain1/glaucoma-datasets'
OUTPUT_PATH   = '/kaggle/working/'

DISEASES = ['RD', 'Glaucome', 'DMLA']

print("Chemins configurés ✓")


Chemins configurés ✓


---
## Diagnostic — Inspection de la Structure des Datasets

Exploration de l'arborescence réelle des 6 datasets avant tout chargement.
Cette étape vérifie que tous les datasets sont correctement montés sur Kaggle
et identifie les chemins exacts vers les images et fichiers CSV.




In [ ]:
# A lancer EN PREMIER pour voir la structure exacte des datasets

def explore_dataset(name, path, max_files=5):
    import pandas as pd
    print("\n" + "="*50)
    print(f"DATASET : {name}")
    print(f"Path    : {path}")
    print("="*50)
    if not os.path.exists(path):
        print("  Chemin introuvable !")
        return
    for root, dirs, files in os.walk(path):
        level = root.replace(path, '').count(os.sep)
        indent = '  ' * level
        print(f"{indent}{os.path.basename(root)}/")
        if level < 3:
            sub = '  ' * (level + 1)
            for f in files[:max_files]:
                print(f"{sub}{f}")
            if len(files) > max_files:
                print(f"{sub}... ({len(files)} fichiers au total)")
        for f in files:
            if f.endswith(('.csv', '.CSV', '.xlsx')):
                fpath = os.path.join(root, f)
                try:
                    df_tmp = pd.read_csv(fpath, nrows=2)
                    print(f"  Colonnes : {df_tmp.columns.tolist()}")
                    print(f"  Exemple  : {df_tmp.iloc[0].to_dict()}")
                except Exception as e:
                    print(f"  Erreur : {e}")

explore_dataset('ODIR-5K',  ODIR_PATH)
explore_dataset('APTOS',    APTOS_PATH)
explore_dataset('RFMiD',    RFMID_PATH)
explore_dataset('GLAUCOMA', GLAUCOMA_PATH)



DATASET : ODIR-5K
Path    : /kaggle/input/datasets/andrewmvd/ocular-disease-recognition-odir5k
ocular-disease-recognition-odir5k/
  full_df.csv
  Colonnes : ['ID', 'Patient Age', 'Patient Sex', 'Left-Fundus', 'Right-Fundus', 'Left-Diagnostic Keywords', 'Right-Diagnostic Keywords', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O', 'filepath', 'labels', 'target', 'filename']
  Exemple  : {'ID': 0, 'Patient Age': 69, 'Patient Sex': 'Female', 'Left-Fundus': '0_left.jpg', 'Right-Fundus': '0_right.jpg', 'Left-Diagnostic Keywords': 'cataract', 'Right-Diagnostic Keywords': 'normal fundus', 'N': 0, 'D': 0, 'G': 0, 'C': 1, 'A': 0, 'H': 0, 'M': 0, 'O': 0, 'filepath': '../input/ocular-disease-recognition-odir5k/ODIR-5K/Training Images/0_right.jpg', 'labels': "['N']", 'target': '[1, 0, 0, 0, 0, 0, 0, 0]', 'filename': '0_right.jpg'}
  preprocessed_images/
    3419_left.jpg
    4176_right.jpg
    3370_left.jpg
    1255_right.jpg
    660_left.jpg
    ... (6392 fichiers au total)
  ODIR-5K/
    ODIR-5K/
      d

---
## M1 — Chargement & Fusion des 6 Datasets

### Dataset 1 / 6 — ODIR-5K

| Caractéristique | Valeur |
|-----------------|--------|
| Pathologies cibles | RD (colonne D), Glaucome (G), DMLA (A) |
| Volume | 10 000 images (5 000 patients × 2 yeux) |
| Format labels | CSV `full_df.csv` — colonnes binaires par pathologie |
| Images | Dossier `Training Images/` |


In [8]:
# CELLULE 2 — Charger ODIR-5K
ODIR_IMG_DIR = os.path.join(ODIR_PATH, 'ODIR-5K', 'ODIR-5K', 'Training Images')
odir_csv     = os.path.join(ODIR_PATH, 'full_df.csv')

df_odir_raw = pd.read_csv(odir_csv)
print(f"ODIR brut : {len(df_odir_raw)} lignes")
print(f"Colonnes  : {df_odir_raw.columns.tolist()}")
print(f"Images    : {ODIR_IMG_DIR}")


ODIR brut : 6392 lignes
Colonnes  : ['ID', 'Patient Age', 'Patient Sex', 'Left-Fundus', 'Right-Fundus', 'Left-Diagnostic Keywords', 'Right-Diagnostic Keywords', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O', 'filepath', 'labels', 'target', 'filename']
Images    : /kaggle/input/datasets/andrewmvd/ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Training Images


### Extraction des labels ODIR-5K

Les colonnes `D` (Rétinopathie diabétique), `G` (Glaucome) et `A` (DMLA)
sont extraites pour chaque image (œil gauche `Left-Fundus` + œil droit `Right-Fundus`).
Le `patient_id` est conservé pour le split anti-fuite de données.


In [9]:
# CELLULE 3 — Extraire labels ODIR-5K
rows = []
for _, row in df_odir_raw.iterrows():
    patient_id = str(row['ID'])
    rd       = int(row.get('D', 0))
    glaucome = int(row.get('G', 0))
    dmla     = int(row.get('A', 0))

    for col in ['Left-Fundus', 'Right-Fundus']:
        fname    = str(row.get(col, ''))
        img_path = os.path.join(ODIR_IMG_DIR, fname)
        if os.path.exists(img_path):
            rows.append({
                'image_path': img_path,
                'patient_id': patient_id,
                'source':     'ODIR',
                'RD': rd, 'Glaucome': glaucome, 'DMLA': dmla
            })

df_odir = pd.DataFrame(rows)
print(f"ODIR-5K : {len(df_odir)} images chargées")
print(f"  RD      : {df_odir['RD'].sum()}")
print(f"  Glaucome: {df_odir['Glaucome'].sum()}")
print(f"  DMLA    : {df_odir['DMLA'].sum()}")


ODIR-5K : 12784 images chargées
  RD      : 4246
  Glaucome: 794
  DMLA    : 638


### Dataset 2 / 6 — APTOS 2019 (Rétinopathie Diabétique)

| Caractéristique | Valeur |
|-----------------|--------|
| Pathologie cible | RD uniquement |
| Volume | 3 662 images |
| Labellisation | Grade ≥ 1 → RD = 1 (binarisation depuis grade 0–4) |
| Prétraitement | Images avec filtre Gaussien, 224×224 pixels |


In [10]:
# CELLULE 4 — Charger APTOS
aptos_base = os.path.join(APTOS_PATH, 'gaussian_filtered_images', 'gaussian_filtered_images')
aptos_csv  = os.path.join(APTOS_PATH, 'train.csv')

diag_to_folder = {0: 'No_DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferate_DR'}

df_aptos_raw = pd.read_csv(aptos_csv)
rows_aptos = []
for _, row in df_aptos_raw.iterrows():
    diag     = int(row['diagnosis'])
    img_id   = str(row['id_code'])
    folder   = diag_to_folder[diag]
    img_path = os.path.join(aptos_base, folder, img_id + '.png')
    if os.path.exists(img_path):
        rows_aptos.append({
            'image_path': img_path,
            'patient_id': 'APTOS_' + img_id,
            'source':     'APTOS',
            'RD':         1 if diag > 0 else 0,
            'Glaucome':   0,
            'DMLA':       0
        })

df_aptos = pd.DataFrame(rows_aptos)
print(f"APTOS : {len(df_aptos)} images chargées")
print(f"  RD positif : {df_aptos['RD'].sum()}")
print(f"  Normal     : {(df_aptos['RD'] == 0).sum()}")


APTOS : 3662 images chargées
  RD positif : 1857
  Normal     : 1805


### Dataset 3 / 6 — RFMiD (Rétinopathie Diabétique + DMLA)

| Caractéristique | Valeur |
|-----------------|--------|
| Pathologies cibles | RD, DMLA |
| Splits inclus | Training + Evaluation + Test |
| Volume total | ~1 920 images |
| Format labels | 3 fichiers CSV (un par split) |


In [11]:
# CELLULE 4b — Charger RFMiD (RD + DMLA)
rfmid_splits = [
    ('Training_Set/Training_Set', 'RFMiD_Training_Labels.csv',   'Training'),
    ('Evaluation_Set/Evaluation_Set', 'RFMiD_Validation_Labels.csv', 'Validation'),
    ('Test_Set/Test_Set',         'RFMiD_Testing_Labels.csv',    'Test'),
]

rows_rfmid = []
for split_dir, csv_name, img_folder in rfmid_splits:
    csv_path = os.path.join(RFMID_PATH, split_dir, csv_name)
    img_dir  = os.path.join(RFMID_PATH, split_dir, img_folder)
    if not os.path.exists(csv_path):
        print(f"⚠ CSV introuvable : {csv_path}")
        continue
    df_split = pd.read_csv(csv_path)
    for _, row in df_split.iterrows():
        for ext in ['.png', '.jpg']:
            img_path = os.path.join(img_dir, f"{int(row['ID'])}{ext}")
            if os.path.exists(img_path):
                rows_rfmid.append({
                    'image_path': img_path,
                    'patient_id': f"RFMID_{int(row['ID'])}",
                    'source':     'RFMID',
                    'RD':         int(row['DR']),
                    'Glaucome':   0,
                    'DMLA':       int(row['ARMD'])
                })
                break

df_rfmid = pd.DataFrame(rows_rfmid)
print(f"RFMiD : {len(df_rfmid)} images chargées")
print(f"  RD   positif : {df_rfmid['RD'].sum()}")
print(f"  DMLA positif : {df_rfmid['DMLA'].sum()}")


RFMiD : 3200 images chargées
  RD   positif : 632
  DMLA positif : 169


### Dataset 4 / 6 — REFUGE (Glaucome)

| Caractéristique | Valeur |
|-----------------|--------|
| Pathologie cible | Glaucome uniquement |
| Volume | 1 200 images (split train uniquement — labels certifiés) |
| Labellisation | Nom de fichier commence par "g" → Glaucome = 1 |


In [16]:
# CELLULE 5 — Charger REFUGE (train uniquement — labels fiables)
refuge_img_dir = os.path.join(GLAUCOMA_PATH, 'REFUGE', 'train', 'Images')

rows_refuge = []
if os.path.exists(refuge_img_dir):
    for fname in os.listdir(refuge_img_dir):
        if not fname.lower().endswith(('.jpg', '.png', '.jpeg')):
            continue
        label = 1 if fname.lower().startswith('g') else 0
        rows_refuge.append({
            'image_path': os.path.join(refuge_img_dir, fname),
            'patient_id': 'REFUGE_' + fname,
            'source': 'REFUGE',
            'RD': 0, 'Glaucome': label, 'DMLA': 0
        })

df_refuge = pd.DataFrame(rows_refuge)
print(f"REFUGE : {len(df_refuge)} images chargées")
print(f"  Glaucome : {df_refuge['Glaucome'].sum()}")
print(f"  Normal   : {(df_refuge['Glaucome']==0).sum()}")


REFUGE : 400 images chargées
  Glaucome : 40
  Normal   : 360


### Datasets 5 / 6 & 6 / 6 — G1020 + ORIGA (Glaucome)

| Dataset | Volume | Source des labels |
|---------|--------|-------------------|
| G1020 | 1 020 images | `G1020.csv` — colonne `binaryLabels` |
| ORIGA | ~650 images | `ORIGA-light.xlsx` — colonne `Glaucoma` |

Ces deux datasets complètent la représentation du Glaucome,
pathologie sous-représentée dans les datasets généralistes.


In [17]:
# CELLULE 5b — Charger G1020 + ORIGA (Glaucome)

# G1020
g1020_base    = os.path.join(GLAUCOMA_PATH, 'G1020')
g1020_csv     = os.path.join(g1020_base, 'G1020.csv')
g1020_img_dir = os.path.join(g1020_base, 'Images_Square')

df_g1020_raw = pd.read_csv(g1020_csv)
rows_g = []
for _, row in df_g1020_raw.iterrows():
    img_path = os.path.join(g1020_img_dir, str(row['imageID']))
    if os.path.exists(img_path):
        rows_g.append({
            'image_path': img_path,
            'patient_id': str(row['imageID']),
            'source': 'G1020',
            'RD': 0, 'Glaucome': int(row['binaryLabels']), 'DMLA': 0
        })
df_g1020 = pd.DataFrame(rows_g)
print(f"G1020 : {len(df_g1020)} images, {df_g1020['Glaucome'].sum()} glaucome")

# ORIGA
origa_base    = os.path.join(GLAUCOMA_PATH, 'ORIGA')
origa_csv     = os.path.join(origa_base, 'OrigaList.csv')
origa_img_dir = os.path.join(origa_base, 'Images_Square')



df_origa_raw = pd.read_csv(origa_csv)
rows_o = []
for _, row in df_origa_raw.iterrows():
    fname    = os.path.basename(str(row['Filename']))
    img_path = os.path.join(origa_img_dir, fname)
    if os.path.exists(img_path):
        rows_o.append({
            'image_path': img_path,
            'patient_id': fname,
            'source': 'ORIGA',
            'RD': 0, 'Glaucome': int(row['Glaucoma']), 'DMLA': 0
        })
df_origa = pd.DataFrame(rows_o)
print(f"ORIGA : {len(df_origa)} images, {df_origa['Glaucome'].sum()} glaucome")

df_g1020_origa = pd.concat([df_g1020, df_origa], ignore_index=True)
print(f"Total G1020+ORIGA : {len(df_g1020_origa)} images")


G1020 : 1020 images, 296 glaucome
ORIGA : 650 images, 168 glaucome
Total G1020+ORIGA : 1670 images


### Fusion des 6 Datasets

Concaténation des 6 sources en un DataFrame unifié avec les colonnes :
`image_path`, `RD`, `Glaucome`, `DMLA`, `patient_id`, `source`.

Les doublons (même chemin d'image) sont supprimés.


In [18]:
df_all = pd.concat(
    [df_odir, df_aptos, df_refuge, df_rfmid, df_g1020, df_origa],
    ignore_index=True
)
df_all = df_all.drop_duplicates(subset='image_path')

print(f"Total fusionné : {len(df_all)} images")
print(df_all['source'].value_counts())


Total fusionné : 15648 images
source
ODIR      6716
APTOS     3662
RFMID     3200
G1020     1020
ORIGA      650
REFUGE     400
Name: count, dtype: int64


### Vérification Qualité des Images

Contrôle d'intégrité de chaque image du dataset fusionné :
- Existence du fichier sur disque
- Lisibilité par OpenCV (détection des images corrompues)




In [19]:
# Un vrai pipeline ne lance pas l'entraînement sur des images
# corrompues ou manquantes. On vérifie ici que chaque chemin
# existe ET que l'image est lisible par OpenCV.
import cv2

print("Vérification qualité des images...")
valid_mask = []
for path in df_all['image_path']:
    if not os.path.exists(path):
        valid_mask.append(False)
        continue
    img = cv2.imread(path)
    valid_mask.append(img is not None and img.shape[0] > 0)

n_before = len(df_all)
df_all   = df_all[valid_mask].reset_index(drop=True)
n_after  = len(df_all)
print(f"  Images avant vérification : {n_before}")
print(f"  Images corrompues/manquantes supprimées : {n_before - n_after}")
print(f"  Images valides retenues : {n_after}")

# Résumé de la distribution finale par maladie et par source
print(f"\n{'─'*50}")
print("DISTRIBUTION FINALE")
print(f"{'─'*50}")
print(f"\nPar source :")
print(df_all['source'].value_counts().to_string())
print(f"\nPar maladie (positifs / total) :")
for d in DISEASES:
    pos   = df_all[d].sum()
    total = len(df_all)
    pct   = 100 * pos / total
    print(f"  {d:10s} : {pos:5d} positifs / {total} images ({pct:.1f}%)")
print(f"\n  Normal (aucune maladie) : {((df_all['RD']==0)&(df_all['Glaucome']==0)&(df_all['DMLA']==0)).sum()} images")
print(f"  Multi-label             : {((df_all[DISEASES].sum(axis=1))>1).sum()} images")


Vérification qualité des images...
  Images avant vérification : 15648
  Images corrompues/manquantes supprimées : 0
  Images valides retenues : 15648

──────────────────────────────────────────────────
DISTRIBUTION FINALE
──────────────────────────────────────────────────

Par source :
source
ODIR      6716
APTOS     3662
RFMID     3200
G1020     1020
ORIGA      650
REFUGE     400

Par maladie (positifs / total) :
  RD         :  4699 positifs / 15648 images (30.0%)
  Glaucome   :   916 positifs / 15648 images (5.9%)
  DMLA       :   495 positifs / 15648 images (3.2%)

  Normal (aucune maladie) : 9652 images
  Multi-label             : 114 images

Dataset fusionné : 15648 images au total

Distribution par source :
source
ODIR      6716
APTOS     3662
RFMID     3200
G1020     1020
ORIGA      650
REFUGE     400
Name: count, dtype: int64

Distribution des maladies :
  RD       : 4699 positifs / 15648 images
  Glaucome : 916 positifs / 15648 images
  DMLA     : 495 positifs / 15648 images

---
## Analyse Exploratoire des Données (EDA)

### Distribution des Pathologies dans le Dataset Fusionné

Visualisation de la répartition des 3 pathologies cibles dans l'ensemble fusionné,
avec identification des cas multi-pathologiques (co-occurrences RD + Glaucome + DMLA).


In [ ]:
# Masques necessaires pour la visualisation
normal_mask = (df_all['RD'] == 0) & (df_all['Glaucome'] == 0) & (df_all['DMLA'] == 0)
multi_mask  = (df_all[DISEASES].sum(axis=1) > 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Distribution par maladie
disease_counts = {
    'Normal (aucune)': normal_mask.sum(),
    'RD': df_all['RD'].sum(),
    'Glaucome': df_all['Glaucome'].sum(),
    'DMLA': df_all['DMLA'].sum(),
    'Multi-label': multi_mask.sum(),
}
axes[0].bar(disease_counts.keys(), disease_counts.values(),
            color=['#27B2C4', '#E86B1F', '#C0392B', '#8E44AD', '#2E8648'])
axes[0].set_title('Distribution des classes')
axes[0].set_ylabel('Nombre d\'images')
axes[0].tick_params(axis='x', rotation=15)
for i, (k, v) in enumerate(disease_counts.items()):
    axes[0].text(i, v + 20, str(v), ha='center', fontsize=10)

# Distribution par source
source_counts = df_all['source'].value_counts()
pie_colors = ['#27B2C4', '#E86B1F', '#2E8648', '#8E44AD', '#C0392B', '#F39C12']
axes[1].pie(source_counts.values, labels=source_counts.index,
                autopct='%1.1f%%',
                colors=pie_colors[:len(source_counts)])
axes[1].set_title('Distribution par source')

# Heatmap co-occurrence maladies
co_matrix = pd.DataFrame({
    'RD':      [df_all[(df_all['RD']==1) & (df_all['RD']==1)].shape[0],
                df_all[(df_all['RD']==1) & (df_all['Glaucome']==1)].shape[0],
                df_all[(df_all['RD']==1) & (df_all['DMLA']==1)].shape[0]],
    'Glaucome':[df_all[(df_all['Glaucome']==1) & (df_all['RD']==1)].shape[0],
                df_all[(df_all['Glaucome']==1) & (df_all['Glaucome']==1)].shape[0],
                df_all[(df_all['Glaucome']==1) & (df_all['DMLA']==1)].shape[0]],
    'DMLA':    [df_all[(df_all['DMLA']==1) & (df_all['RD']==1)].shape[0],
                df_all[(df_all['DMLA']==1) & (df_all['Glaucome']==1)].shape[0],
                df_all[(df_all['DMLA']==1) & (df_all['DMLA']==1)].shape[0]],
}, index=['RD', 'Glaucome', 'DMLA'])
sns.heatmap(co_matrix, annot=True, fmt='d', cmap='Blues', ax=axes[2])
axes[2].set_title('Co-occurrence des maladies')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'distribution_maladies.png'), dpi=150)
plt.show()

---
## M3 — Split Stratifié Train / Val / Test (70 / 15 / 15)

### Stratégie anti-fuite de données

| Source | Stratégie de split |
|--------|-------------------|
| **ODIR-5K** | Split au niveau **patient** (œil gauche et droit restent ensemble) |
| **Autres datasets** | Split au niveau image (patient_id unique par image) |

> Le split est stratifié pour maintenir la même proportion de pathologies dans chaque ensemble.


In [22]:
from sklearn.model_selection import train_test_split

# Séparer ODIR (patient_id réel) des autres (patient_id unique par image)
df_odir_only = df_all[df_all['source'] == 'ODIR'].copy()
df_others    = df_all[df_all['source'] != 'ODIR'].copy()

# ODIR : split par patient (éviter fuite données)
patients = df_odir_only['patient_id'].unique()
patients_train, patients_temp = train_test_split(patients, test_size=0.30, random_state=42)
patients_val, patients_test   = train_test_split(patients_temp, test_size=0.50, random_state=42)

odir_train = df_odir_only[df_odir_only['patient_id'].isin(patients_train)]
odir_val   = df_odir_only[df_odir_only['patient_id'].isin(patients_val)]
odir_test  = df_odir_only[df_odir_only['patient_id'].isin(patients_test)]

# Autres datasets : split simple 70/15/15
others_train, others_temp = train_test_split(df_others, test_size=0.30, random_state=42)
others_val, others_test   = train_test_split(others_temp, test_size=0.50, random_state=42)

# Fusion finale
df_train = pd.concat([odir_train, others_train], ignore_index=True).sample(frac=1, random_state=42)
df_val   = pd.concat([odir_val,   others_val],   ignore_index=True).sample(frac=1, random_state=42)
df_test  = pd.concat([odir_test,  others_test],  ignore_index=True).sample(frac=1, random_state=42)

print(f"Train : {len(df_train)} images")
print(f"Val   : {len(df_val)} images")
print(f"Test  : {len(df_test)} images")
print(f"\nDistribution Train :")
for d in DISEASES:
    print(f"  {d} : {df_train[d].sum()} positifs")

Train : 10952 images
Val   : 2348 images
Test  : 2348 images

Distribution Train :
  RD : 3273 positifs
  Glaucome : 614 positifs
  DMLA : 358 positifs


### Sauvegarde des Splits en CSV

Les trois ensembles sont sauvegardés dans `/kaggle/working/` :
`train.csv`, `val.csv`, `test.csv` — utilisés par tous les modèles.


In [23]:
df_train.to_csv(os.path.join(OUTPUT_PATH, 'train.csv'), index=False)
df_val.to_csv(  os.path.join(OUTPUT_PATH, 'val.csv'),   index=False)
df_test.to_csv( os.path.join(OUTPUT_PATH, 'test.csv'),  index=False)
df_all.to_csv(  os.path.join(OUTPUT_PATH, 'all.csv'),   index=False)

print("CSVs sauvegardés :")
print(f"  /kaggle/working/train.csv — {len(df_train)} lignes")
print(f"  /kaggle/working/val.csv   — {len(df_val)} lignes")
print(f"  /kaggle/working/test.csv  — {len(df_test)} lignes")
print("\nPréparation des données terminée ✓")

CSVs sauvegardés :
  /kaggle/working/train.csv — 10952 lignes
  /kaggle/working/val.csv   — 2348 lignes
  /kaggle/working/test.csv  — 2348 lignes

Préparation des données terminée ✓


---
## M2 — Prétraitement des Images & Augmentation

### Pipeline de prétraitement (tous ensembles)

1. **Resize** → 224×224 pixels
2. **Normalisation** → statistiques ImageNet (μ = 0.485/0.456/0.406 · σ = 0.229/0.224/0.225)

### Augmentation (train set uniquement)

| Transformation | Paramètres | Objectif |
|----------------|-----------|---------|
| HorizontalFlip | p=0.5 | Invariance gauche/droite |
| VerticalFlip | p=0.5 | Invariance haut/bas |
| Rotation | ±15°, p=0.7 | Robustesse à l'orientation |
| CLAHE | p=0.3 | Variation de contraste |
| GaussNoise | p=0.3 | Robustesse au bruit |
| ColorJitter | brightness/contrast, p=0.3 | Variation d'éclairage |


In [24]:
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from torch.utils.data import Dataset, DataLoader

# Ces constantes sont déjà définies dans la cellule 0 (kaggle_cells.py)
# On les redéfinit ici pour que ce fichier soit autonome sur Kaggle
IMG_SIZE    = 224
BATCH_SIZE  = 16
NUM_CLASSES = 3
DISEASES    = ['RD', 'Glaucome', 'DMLA']
SEED        = 42

# Seeds PyTorch
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

def preprocess_clahe(img_bgr):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    channels = cv2.split(img_rgb)
    channels = [clahe.apply(c) for c in channels]
    return cv2.merge(channels)

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=15, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.1, rotate_limit=0, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
    A.GaussNoise(var_limit=(5, 20), p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

print("Preprocessing configuré ✓")

Preprocessing configuré ✓


### Dataset PyTorch — FundusDataset

Classe `Dataset` personnalisée pour le chargement multi-label.
Chaque item retourne un tuple `(image_tensor, label_vector)` où
`label_vector = [RD, Glaucome, DMLA]` (3 valeurs binaires flottantes).


In [27]:
class FundusDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        img_bgr = cv2.imread(row['image_path'])
        if img_bgr is None:
            img_bgr = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        img_rgb = preprocess_clahe(img_bgr)

        if self.transform:
            img = self.transform(image=img_rgb)['image']
        else:
            img = torch.tensor(img_rgb).permute(2, 0, 1).float() / 255.0

        # Labels multi-label : [RD, Glaucome, DMLA]
        labels = torch.tensor(
            [row['RD'], row['Glaucome'], row['DMLA']], dtype=torch.float32
        )
        return img, labels

# DataLoaders
train_loader = DataLoader(
    FundusDataset(df_train, train_transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    FundusDataset(df_val, val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    FundusDataset(df_test, val_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

print(f"Train loader : {len(train_loader)} batches")
print(f"Val loader   : {len(val_loader)} batches")
print(f"Test loader  : {len(test_loader)} batches")

Train loader : 685 batches
Val loader   : 147 batches
Test loader  : 147 batches


---
## M4 — Entraînement des Modèles CNN (Keras)

### Architecture commune (DenseNet121 · EfficientNetV2S · EfficientNetB3)

```
Backbone pré-entraîné (ImageNet)
    ↓
Global Average Pooling 2D
    ↓
Dense(256, ReLU) + Dropout(0.5)
    ↓
Dense(3, Sigmoid)   →   [P(RD), P(Glaucome), P(DMLA)]
```

**Fonction de perte :** Weighted Binary Cross-Entropy
(pondération inverse de la fréquence de chaque pathologie)

### Stratégie d'entraînement en 2 phases

| Phase | Backbone | Learning Rate | Epochs |
|-------|---------|--------------|--------|
| Phase 1 — Warm-up | Gelé | 1e-3 | 10 |
| Phase 2 — Fine-tuning | Dégelé | 1e-5 | 30 |

> **Callbacks** : EarlyStopping (patience=5) · ModelCheckpoint (val AUC) · ReduceLROnPlateau


In [29]:
import tensorflow as tf
import gc
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import DenseNet121, EfficientNetV2S, EfficientNetB3
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

MODELS_DIR = '/kaggle/working/models'
os.makedirs(MODELS_DIR, exist_ok=True)

def weighted_bce(pos_weights):
    """
    Weighted binary crossentropy pour classification multi-label.
    pos_weights[i] = neg_i / pos_i : upweight les exemples positifs par maladie.
    Les negatifs gardent un poids 1.0.
    Compatible avec les generateurs Keras (pas de class_weight Keras).
    """
    weights_tensor = tf.constant(pos_weights, dtype=tf.float32)
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce = -(y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        w = y_true * weights_tensor + (1.0 - y_true)
        return tf.reduce_mean(bce * w)
    return loss

def build_keras_model(base_model_fn, name):
    base = base_model_fn(weights='imagenet', include_top=False,
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(NUM_CLASSES, activation='sigmoid')(x)
    model = Model(inputs=base.input, outputs=output, name=name)
    return model

def train_keras_model(model, name, df_train, df_val, epochs=20):
    pos_weights = []
    print(f"Poids classes pour {name} :")
    for disease in DISEASES:
        pos = df_train[disease].sum()
        neg = len(df_train) - pos
        w = neg / pos if pos > 0 else 1.0
        pos_weights.append(w)
        print(f"  {disease}: pos={int(pos)}, neg={int(neg)}, poids={w:.2f}")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=weighted_bce(pos_weights),
        metrics=[tf.keras.metrics.AUC(multi_label=True, name='auc')]
    )

    keras_train_tf = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=15, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
        A.GaussNoise(var_limit=(5, 20), p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    keras_val_tf = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])

    def load_img_keras(path, transform):
        img = cv2.imread(path)
        if img is None:
            return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        img = preprocess_clahe(img)
        return transform(image=img)['image'].astype(np.float32)

    def generator(df, augment=False):
        transform = keras_train_tf if augment else keras_val_tf
        while True:
            df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
            for start in range(0, len(df), BATCH_SIZE):
                batch = df.iloc[start:start+BATCH_SIZE]
                imgs, labels = [], []
                for _, row in batch.iterrows():
                    imgs.append(load_img_keras(row['image_path'], transform))
                    labels.append([row['RD'], row['Glaucome'], row['DMLA']])
                yield np.array(imgs), np.array(labels, dtype=np.float32)

    callbacks = [
        ModelCheckpoint(f'{MODELS_DIR}/{name}_best.keras',
                        monitor='val_auc', mode='max',
                        save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_auc', mode='max', patience=10, verbose=1),
        ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5,
                          patience=5, verbose=1)
    ]

    steps_train = len(df_train) // BATCH_SIZE
    steps_val   = len(df_val)   // BATCH_SIZE

    history = model.fit(
        generator(df_train, augment=True),
        steps_per_epoch=steps_train,
        validation_data=generator(df_val),
        validation_steps=steps_val,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    return history

print("Fonctions Keras definies OK")

Fonctions Keras definies OK


### DenseNet121 — Entraînement


In [19]:
print("="*50)
print("Entraînement DenseNet121")
print("="*50)
model = build_keras_model(DenseNet121, 'DenseNet121')
hist_densenet = train_keras_model(model, 'DenseNet121', df_train, df_val)
del model
tf.keras.backend.clear_session()
gc.collect()
print("DenseNet121 terminé ✓")


Entraînement DenseNet121


I0000 00:00:1782715107.636142      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782715107.642167      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Poids classes pour DenseNet121 :
  RD: pos=3273, neg=7679, poids=2.35
  Glaucome: pos=614, neg=10338, poids=16.84
  DMLA: pos=358, neg=10594, poids=29.59
Epoch 1/20


I0000 00:00:1782715172.002292     142 service.cc:152] XLA service 0x7992e4003200 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782715172.002343     142 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1782715172.002350     142 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1782715182.286978     142 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1782715272.144703     142 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.7262 - loss: 1.0837
Epoch 1: val_auc improved from None to 0.87691, saving model to /kaggle/working/models/DenseNet121_best.keras

Epoch 1: finished saving model to /kaggle/working/models/DenseNet121_best.keras
684/684 ━━━━━━━━━━━━━━━━━━━━ 1037s 1s/step - auc: 0.7801 - loss: 0.9762 - val_auc: 0.8769 - val_loss: 0.8304 - learning_rate: 1.0000e-04
Epoch 2/20
684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.8355 - loss: 0.8317
Epoch 2: val_auc improved from 0.87691 to 0.89384, saving model to /kaggle/working/models/DenseNet121_best.keras

Epoch 2: finished saving model to /kaggle/working/models/DenseNet121_best.keras
684/684 ━━━━━━━━━━━━━━━━━━━━ 984s 1s/step - auc: 0.8415 - loss: 0.8249 - val_auc: 0.8938 - val_loss: 0.7334 - learning_rate: 1.0000e-04
Epoch 3/20
684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.8711 - loss: 0.7541
Epoch 3: val_auc improved from 0.89384 to 0.89458, saving model to /kaggle/working/models/DenseNet121_best.keras


### Fonction d'Évaluation sur le Test Set (Keras)

Définition de `evaluate_keras_test` : charge un modèle Keras depuis son fichier `.keras`,
prédit sur le test set et retourne les métriques (AUC, sensibilité, spécificité, matrice de confusion)
pour chacune des 3 pathologies.


In [22]:
from sklearn.metrics import roc_auc_score, confusion_matrix

def evaluate_keras_test(model_path, name):
    model = tf.keras.models.load_model(
        model_path,
        custom_objects={'loss': weighted_bce}
    )
    
    keras_val_tf = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    
    imgs, labels_list = [], []
    for _, row in df_test.iterrows():
        img = cv2.imread(row['image_path'])
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        else:
            img = preprocess_clahe(img)
        img = keras_val_tf(image=img)['image'].astype(np.float32)
        imgs.append(img)
        labels_list.append([row['RD'], row['Glaucome'], row['DMLA']])
    
    imgs   = np.array(imgs)
    labels = np.array(labels_list, dtype=np.float32)
    preds  = model.predict(imgs, batch_size=32, verbose=0)
    
    results = {}
    print(f"\n{'='*50}")
    print(f"Évaluation {name} — Test set")
    print(f"{'='*50}")
    for i, disease in enumerate(DISEASES):
        y_true = labels[:, i]
        y_pred = preds[:, i]
        auc    = roc_auc_score(y_true, y_pred)
        y_bin  = (y_pred >= 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_bin, labels=[0,1]).ravel()
        sens = tp / (tp + fn + 1e-8)
        spec = tn / (tn + fp + 1e-8)
        f1   = 2*tp / (2*tp + fp + fn + 1e-8)
        print(f"{disease}: AUC={auc:.4f} | Sens={sens:.4f} | Spec={spec:.4f} | F1={f1:.4f}")
        results[disease] = {'AUC': auc, 'Sensibilité': sens, 'Spécificité': spec, 'F1': f1}
    
    mean_auc = np.mean([results[d]['AUC'] for d in DISEASES])
    results['Mean'] = {'AUC': mean_auc}
    print(f"Mean AUC = {mean_auc:.4f}")
    return results, preds

results_densenet, preds_densenet = evaluate_keras_test(
    f'{MODELS_DIR}/DenseNet121_best.keras', 'DenseNet121'
)



Évaluation DenseNet121 — Test set
RD: AUC=0.9100 | Sens=0.7938 | Spec=0.8423 | F1=0.7424
Glaucome: AUC=0.9191 | Sens=0.8000 | Spec=0.8751 | F1=0.4070
DMLA: AUC=0.8955 | Sens=0.8000 | Spec=0.8182 | F1=0.1832
Mean AUC = 0.9082


### EfficientNetV2S — Entraînement


In [23]:
print("="*50)
print("Entraînement EfficientNetV2S")
print("="*50)
model = build_keras_model(EfficientNetV2S, 'EfficientNetV2S')
hist_effnetv2 = train_keras_model(model, 'EfficientNetV2S', df_train, df_val)
del model
tf.keras.backend.clear_session()
gc.collect()
print("EfficientNetV2S terminé ✓")


Entraînement EfficientNetV2S
82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Poids classes pour EfficientNetV2S :
  RD: pos=3273, neg=7679, poids=2.35
  Glaucome: pos=614, neg=10338, poids=16.84
  DMLA: pos=358, neg=10594, poids=29.59
Epoch 1/20


2026-06-29 11:27:01.317609: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 11:27:01.453247: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 11:27:02.280665: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 11:27:02.418575: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 11:27:03.847022: E external/local_xla/xla/stream_

684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.7372 - loss: 1.0231
Epoch 1: val_auc improved from None to 0.88505, saving model to /kaggle/working/models/EfficientNetV2S_best.keras

Epoch 1: finished saving model to /kaggle/working/models/EfficientNetV2S_best.keras
684/684 ━━━━━━━━━━━━━━━━━━━━ 1009s 1s/step - auc: 0.7980 - loss: 0.9270 - val_auc: 0.8850 - val_loss: 0.7663 - learning_rate: 1.0000e-04
Epoch 2/20


2026-06-29 11:42:32.535499: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 11:42:32.670665: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 11:42:33.476581: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 11:42:33.614013: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 11:42:35.003504: E external/local_xla/xla/stream_

684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.8545 - loss: 0.7913
Epoch 2: val_auc improved from 0.88505 to 0.89414, saving model to /kaggle/working/models/EfficientNetV2S_best.keras

Epoch 2: finished saving model to /kaggle/working/models/EfficientNetV2S_best.keras
684/684 ━━━━━━━━━━━━━━━━━━━━ 939s 1s/step - auc: 0.8609 - loss: 0.7834 - val_auc: 0.8941 - val_loss: 0.7332 - learning_rate: 1.0000e-04
Epoch 3/20
684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.8822 - loss: 0.7299
Epoch 3: val_auc improved from 0.89414 to 0.90909, saving model to /kaggle/working/models/EfficientNetV2S_best.keras

Epoch 3: finished saving model to /kaggle/working/models/EfficientNetV2S_best.keras
684/684 ━━━━━━━━━━━━━━━━━━━━ 897s 1s/step - auc: 0.8845 - loss: 0.7210 - val_auc: 0.9091 - val_loss: 0.6718 - learning_rate: 1.0000e-04
Epoch 4/20
684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.8945 - loss: 0.6971

2026-06-29 12:24:31.905776: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 12:24:32.038979: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 12:24:33.001902: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 12:24:33.141744: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 12:24:34.910021: E external/local_xla/xla/stream_


Epoch 4: val_auc improved from 0.90909 to 0.91418, saving model to /kaggle/working/models/EfficientNetV2S_best.keras

Epoch 4: finished saving model to /kaggle/working/models/EfficientNetV2S_best.keras
684/684 ━━━━━━━━━━━━━━━━━━━━ 865s 1s/step - auc: 0.8977 - loss: 0.6762 - val_auc: 0.9142 - val_loss: 0.6598 - learning_rate: 1.0000e-04
Epoch 5/20
684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.9099 - loss: 0.6343
Epoch 5: val_auc did not improve from 0.91418
684/684 ━━━━━━━━━━━━━━━━━━━━ 882s 1s/step - auc: 0.9066 - loss: 0.6462 - val_auc: 0.9009 - val_loss: 0.7490 - learning_rate: 1.0000e-04
Epoch 6/20
684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.9112 - loss: 0.6108
Epoch 6: val_auc did not improve from 0.91418
684/684 ━━━━━━━━━━━━━━━━━━━━ 876s 1s/step - auc: 0.9124 - loss: 0.6159 - val_auc: 0.8962 - val_loss: 0.8939 - learning_rate: 1.0000e-04
Epoch 7/20
684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 976ms/step - auc: 0.9179 - loss: 0.5765
Epoch 7: val_auc did not improve from 0.91418
684/684 

### Évaluation — EfficientNetV2S sur le Test Set


In [24]:
results_effnetv2, preds_effnetv2 = evaluate_keras_test(
    f'{MODELS_DIR}/EfficientNetV2S_best.keras', 'EfficientNetV2S'
)


2026-06-29 15:51:58.341656: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 15:51:58.482410: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 15:51:59.368609: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 15:51:59.512904: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 15:52:01.028333: E external/local_xla/xla/stream_


Évaluation EfficientNetV2S — Test set
RD: AUC=0.9075 | Sens=0.8277 | Spec=0.7914 | F1=0.7249
Glaucome: AUC=0.9216 | Sens=0.9000 | Spec=0.8314 | F1=0.3768
DMLA: AUC=0.9245 | Sens=0.8167 | Spec=0.8995 | F1=0.2891
Mean AUC = 0.9179


### EfficientNetB3 — Entraînement


In [ ]:
print("="*50)
print("Entraînement EfficientNetB3")
print("="*50)
model = build_keras_model(EfficientNetB3, 'EfficientNetB3')
hist_effnet = train_keras_model(model, 'EfficientNetB3', df_train, df_val)
del model
tf.keras.backend.clear_session()
gc.collect()
print("EfficientNetB3 terminé ✓")


Entraînement EfficientNetB3
43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Poids classes pour EfficientNetB3 :
  RD: pos=3273, neg=7679, poids=2.35
  Glaucome: pos=614, neg=10338, poids=16.84
  DMLA: pos=358, neg=10594, poids=29.59
Epoch 1/20


2026-06-29 15:55:11.831767: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 15:55:11.974563: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 15:55:12.329318: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 15:55:12.478171: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 15:55:13.305200: E external/local_xla/xla/stream_

684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.7246 - loss: 1.0338
Epoch 1: val_auc improved from None to 0.86533, saving model to /kaggle/working/models/EfficientNetB3_best.keras

Epoch 1: finished saving model to /kaggle/working/models/EfficientNetB3_best.keras
684/684 ━━━━━━━━━━━━━━━━━━━━ 1030s 1s/step - auc: 0.7850 - loss: 0.9456 - val_auc: 0.8653 - val_loss: 0.8543 - learning_rate: 1.0000e-04
Epoch 2/20


2026-06-29 16:11:20.115151: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 16:11:20.253694: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 16:11:20.589161: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 16:11:20.737214: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 16:11:21.604700: E external/local_xla/xla/stream_

684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.8494 - loss: 0.8044
Epoch 2: val_auc improved from 0.86533 to 0.88184, saving model to /kaggle/working/models/EfficientNetB3_best.keras

Epoch 2: finished saving model to /kaggle/working/models/EfficientNetB3_best.keras
684/684 ━━━━━━━━━━━━━━━━━━━━ 961s 1s/step - auc: 0.8576 - loss: 0.7893 - val_auc: 0.8818 - val_loss: 0.7550 - learning_rate: 1.0000e-04
Epoch 3/20
684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.8860 - loss: 0.7117
Epoch 3: val_auc improved from 0.88184 to 0.88755, saving model to /kaggle/working/models/EfficientNetB3_best.keras

Epoch 3: finished saving model to /kaggle/working/models/EfficientNetB3_best.keras
684/684 ━━━━━━━━━━━━━━━━━━━━ 878s 1s/step - auc: 0.8867 - loss: 0.7117 - val_auc: 0.8875 - val_loss: 0.8956 - learning_rate: 1.0000e-04
Epoch 4/20
684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.8933 - loss: 0.6875

2026-06-29 16:53:46.285716: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 16:53:46.423396: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 16:53:46.778723: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 16:53:46.927246: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-29 16:53:47.974340: E external/local_xla/xla/stream_


Epoch 4: val_auc improved from 0.88755 to 0.91393, saving model to /kaggle/working/models/EfficientNetB3_best.keras

Epoch 4: finished saving model to /kaggle/working/models/EfficientNetB3_best.keras
684/684 ━━━━━━━━━━━━━━━━━━━━ 894s 1s/step - auc: 0.8928 - loss: 0.6858 - val_auc: 0.9139 - val_loss: 0.6589 - learning_rate: 1.0000e-04
Epoch 5/20
684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.9050 - loss: 0.6460
Epoch 5: val_auc did not improve from 0.91393
684/684 ━━━━━━━━━━━━━━━━━━━━ 877s 1s/step - auc: 0.9039 - loss: 0.6503 - val_auc: 0.8851 - val_loss: 0.9223 - learning_rate: 1.0000e-04
Epoch 6/20
684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.9163 - loss: 0.5946
Epoch 6: val_auc did not improve from 0.91393
684/684 ━━━━━━━━━━━━━━━━━━━━ 870s 1s/step - auc: 0.9143 - loss: 0.6133 - val_auc: 0.9066 - val_loss: 0.8343 - learning_rate: 1.0000e-04
Epoch 7/20
684/684 ━━━━━━━━━━━━━━━━━━━━ 0s 942ms/step - auc: 0.9206 - loss: 0.5685
Epoch 7: val_auc did not improve from 0.91393
684/684 ━━

### Évaluation — EfficientNetB3 sur le Test Set


In [ ]:
results_effnet, preds_effnet = evaluate_keras_test(
    f'{MODELS_DIR}/EfficientNetB3_best.keras', 'EfficientNetB3'
)

---
## M4b — RETFound (ViT-Large) — Entraînement Multi-Label

### Phase 1 — Warm-up (backbone gelé)

Chargement de RETFound avec les poids pré-entraînés sur **1,6 million d'images rétiniennes**.
La tête de classification originale est remplacée par une nouvelle tête multi-label :
`Linear(1024 → 3)` avec activation Sigmoid.

Seule la tête est entraînée en Phase 1 — le backbone est entièrement gelé.

| Paramètre | Valeur |
|-----------|--------|
| Optimiseur | AdamW |
| Scheduler | CosineAnnealingLR |
| Gradient accumulation | ACCUM_STEPS = 16 |


In [30]:
import torch.nn as nn
import torch.nn.functional as F
import types
import timm
from torch.cuda.amp import autocast, GradScaler
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")

model_rf = timm.create_model('vit_large_patch16_224', pretrained=False,
                              num_classes=NUM_CLASSES, global_pool='avg')

def _forward_multilabel(self, x):
    x = x.float()
    B = x.shape[0]
    x = self.patch_embed(x)
    cls_tok = self.cls_token.expand(B, -1, -1)
    x = torch.cat((cls_tok, x), dim=1)
    x = x + self.pos_embed
    x = self.pos_drop(x)
    for blk in self.blocks:
        x = blk(x)
    x = x[:, 1:].mean(dim=1)
    x = self.fc_norm(x)
    return self.head(x)

model_rf.forward = types.MethodType(_forward_multilabel, model_rf)

RETFOUND_WEIGHTS = '/kaggle/input/datasets/kahinaiker/kahina-retfound-weights/RETFound_mae_natureCFP.pth'

if os.path.exists(RETFOUND_WEIGHTS):
    checkpoint = torch.load(RETFOUND_WEIGHTS, map_location='cpu', weights_only=False)
    state = checkpoint['model']
    msg = model_rf.load_state_dict(state, strict=False)
    print(f"Poids chargés : {msg}")
else:
    print("⚠ Poids RETFound non trouvés — entraînement depuis ImageNet")

model_rf = model_rf.to(DEVICE)

for p in model_rf.parameters():
    p.requires_grad = False
for p in model_rf.head.parameters():
    p.requires_grad = True
for p in model_rf.fc_norm.parameters():
    p.requires_grad = True

def compute_class_weights(df):
    weights = []
    for d in DISEASES:
        pos = df[d].sum()
        neg = len(df) - pos
        w_pos = neg / pos if pos > 0 else 1.0
        weights.append(w_pos)
    return torch.tensor(weights, device=DEVICE, dtype=torch.float32)

class_weights = compute_class_weights(df_train)
print(f"Poids : RD={class_weights[0]:.3f}, Glaucome={class_weights[1]:.3f}, DMLA={class_weights[2]:.3f}")

def focal_loss_multilabel(preds, labels, gamma=2.0, weights=None):
    bce = F.binary_cross_entropy_with_logits(preds, labels, reduction='none')
    pt  = torch.exp(-bce)
    loss = (1 - pt) ** gamma * bce
    if weights is not None:
        loss = loss * weights.unsqueeze(0)
    return loss.mean()

def evaluate_multilabel(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            out  = model(imgs)
            probs = torch.sigmoid(out).cpu().numpy()
            all_preds.append(probs)
            all_labels.append(labels.numpy())
    preds  = np.concatenate(all_preds,  axis=0)
    labels = np.concatenate(all_labels, axis=0)
    aucs = []
    for i, d in enumerate(DISEASES):
        if labels[:, i].sum() > 0:
            auc = roc_auc_score(labels[:, i], preds[:, i])
            aucs.append(auc)
            print(f"  {d} AUC : {auc:.4f}")
        else:
            print(f"  {d} : pas d'exemples positifs")
            aucs.append(0.0)
    mean_auc = np.mean(aucs)
    print(f"  Mean AUC : {mean_auc:.4f}")
    return mean_auc, aucs, preds, labels

optimizer_p1 = Adam(
    filter(lambda p: p.requires_grad, model_rf.parameters()), lr=1e-3
)
scheduler_p1 = CosineAnnealingLR(optimizer_p1, T_max=20)
best_auc_p1 = 0.0
patience_p1 = 0
MAX_PATIENCE_P1 = 7

print("\n--- Phase 1 : backbone gelé ---")
for epoch in range(1, 21):
    model_rf.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer_p1.zero_grad()
        out  = model_rf(imgs)
        loss = focal_loss_multilabel(out, labels, weights=class_weights)
        loss.backward()
        optimizer_p1.step()
        total_loss += loss.item()
    scheduler_p1.step()
    print(f"\nEpoch {epoch} | Loss: {total_loss/len(train_loader):.4f}")
    mean_auc, _, _, _ = evaluate_multilabel(model_rf, val_loader)
    if mean_auc > best_auc_p1:
        best_auc_p1 = mean_auc
        torch.save(model_rf.state_dict(), f'{MODELS_DIR}/RETFound_phase1.pth')
        print(f"  ✓ Sauvegardé (Mean AUC={best_auc_p1:.4f})")
        patience_p1 = 0
    else:
        patience_p1 += 1
        if patience_p1 >= MAX_PATIENCE_P1:
            print(f"  Early stopping Phase 1 à epoch {epoch}")
            break

print(f"\nPhase 1 terminée — Best Mean AUC = {best_auc_p1:.4f}")


Device : cuda
Poids chargés : _IncompatibleKeys(missing_keys=['fc_norm.weight', 'fc_norm.bias', 'head.weight', 'head.bias'], unexpected_keys=['mask_token', 'decoder_pos_embed', 'decoder_embed.weight', 'decoder_embed.bias', 'decoder_blocks.0.norm1.weight', 'decoder_blocks.0.norm1.bias', 'decoder_blocks.0.attn.qkv.weight', 'decoder_blocks.0.attn.qkv.bias', 'decoder_blocks.0.attn.proj.weight', 'decoder_blocks.0.attn.proj.bias', 'decoder_blocks.0.norm2.weight', 'decoder_blocks.0.norm2.bias', 'decoder_blocks.0.mlp.fc1.weight', 'decoder_blocks.0.mlp.fc1.bias', 'decoder_blocks.0.mlp.fc2.weight', 'decoder_blocks.0.mlp.fc2.bias', 'decoder_blocks.1.norm1.weight', 'decoder_blocks.1.norm1.bias', 'decoder_blocks.1.attn.qkv.weight', 'decoder_blocks.1.attn.qkv.bias', 'decoder_blocks.1.attn.proj.weight', 'decoder_blocks.1.attn.proj.bias', 'decoder_blocks.1.norm2.weight', 'decoder_blocks.1.norm2.bias', 'decoder_blocks.1.mlp.fc1.weight', 'decoder_blocks.1.mlp.fc1.bias', 'decoder_blocks.1.mlp.fc2.weight'

### Phase 2a — Fine-tuning Sélectif (8 derniers blocs)

Dégel des **8 derniers blocs Transformer** (N_UNFREEZE = 8) + fc_norm + head.
Le reste du backbone reste gelé pour préserver les représentations générales rétiniennes.


In [ ]:
import gc, os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
ACCUM_STEPS = 16
N_UNFREEZE  = 8

# Charger meilleur checkpoint Phase 1
model_rf.load_state_dict(torch.load(f'{MODELS_DIR}/RETFound_phase1.pth', map_location=DEVICE))

# Dégeler les 8 derniers blocs + fc_norm + head
for p in model_rf.parameters():
    p.requires_grad = False
for blk in model_rf.blocks[-N_UNFREEZE:]:
    for p in blk.parameters():
        p.requires_grad = True
for p in model_rf.fc_norm.parameters():
    p.requires_grad = True
for p in model_rf.head.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model_rf.parameters() if p.requires_grad)
print(f"Paramètres entraînables Phase 2 : {trainable/1e6:.1f}M")

# Paramètres séparés backbone/tête
backbone_params = [p for blk in model_rf.blocks[-N_UNFREEZE:] for p in blk.parameters()]
head_params     = list(model_rf.fc_norm.parameters()) + list(model_rf.head.parameters())

optimizer_p2 = AdamW([
    {'params': backbone_params, 'lr': 1e-6},
    {'params': head_params,     'lr': 5e-5},
], weight_decay=0.05)
scheduler_p2 = CosineAnnealingLR(optimizer_p2, T_max=30)
scaler       = GradScaler()

best_auc_p2  = 0.0
patience_p2  = 0
MAX_PATIENCE_P2 = 10

print("\n--- Phase 2 : fine-tuning sélectif ---")
for epoch in range(1, 21):
    model_rf.train()
    total_loss = 0
    optimizer_p2.zero_grad()

    for step, (imgs, labels) in enumerate(train_loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        with autocast():
            out  = model_rf(imgs)
            loss = focal_loss_multilabel(out, labels, weights=class_weights)
            loss = loss / ACCUM_STEPS

        scaler.scale(loss).backward()
        total_loss += loss.item() * ACCUM_STEPS

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.step(optimizer_p2)
            scaler.update()
            optimizer_p2.zero_grad()

    # Flush des gradients résiduels (si len(loader) n'est pas multiple de ACCUM_STEPS)
    if (step + 1) % ACCUM_STEPS != 0:
        scaler.step(optimizer_p2)
        scaler.update()
        optimizer_p2.zero_grad()

    scheduler_p2.step()
    print(f"\nEpoch {epoch} | Loss: {total_loss/len(train_loader):.4f}")
    mean_auc, aucs, _, _ = evaluate_multilabel(model_rf, val_loader)

    if mean_auc > best_auc_p2:
        best_auc_p2 = mean_auc
        torch.save(model_rf.state_dict(), f'{MODELS_DIR}/RETFound_best.pth')
        print(f"  ✓ Sauvegardé (Mean AUC={best_auc_p2:.4f})")
        patience_p2 = 0
    else:
        patience_p2 += 1
        if patience_p2 >= MAX_PATIENCE_P2:
            print(f"  Early stopping Phase 2 à epoch {epoch}")
            break

    gc.collect()
    torch.cuda.empty_cache()

print(f"\nPhase 2 terminée — Best Mean AUC = {best_auc_p2:.4f}")

Paramètres entraînables Phase 2 : 100.8M

--- Phase 2 : fine-tuning sélectif ---

Epoch 1 | Loss: 0.5974
  RD AUC : 0.8779
  Glaucome AUC : 0.9071
  DMLA AUC : 0.8699
  Mean AUC : 0.8850
  ✓ Sauvegardé (Mean AUC=0.8850)

Epoch 2 | Loss: 0.6060
  RD AUC : 0.8781
  Glaucome AUC : 0.9068
  DMLA AUC : 0.8712
  Mean AUC : 0.8854
  ✓ Sauvegardé (Mean AUC=0.8854)

Epoch 3 | Loss: 0.5942
  RD AUC : 0.8783
  Glaucome AUC : 0.9076
  DMLA AUC : 0.8724
  Mean AUC : 0.8861
  ✓ Sauvegardé (Mean AUC=0.8861)

Epoch 4 | Loss: 0.5952
  RD AUC : 0.8785
  Glaucome AUC : 0.9082
  DMLA AUC : 0.8721
  Mean AUC : 0.8862
  ✓ Sauvegardé (Mean AUC=0.8862)

Epoch 5 | Loss: 0.6002
  RD AUC : 0.8782
  Glaucome AUC : 0.9086
  DMLA AUC : 0.8733
  Mean AUC : 0.8867
  ✓ Sauvegardé (Mean AUC=0.8867)

Epoch 6 | Loss: 0.5978
  RD AUC : 0.8786
  Glaucome AUC : 0.9085
  DMLA AUC : 0.8747
  Mean AUC : 0.8873
  ✓ Sauvegardé (Mean AUC=0.8873)

Epoch 7 | Loss: 0.6031
  RD AUC : 0.8789
  Glaucome AUC : 0.9082
  DMLA AUC : 0.8725

KeyboardInterrupt: 

### Phase 2b — Fine-tuning Étendu (12 derniers blocs)

Extension du dégel à **12 blocs Transformer** (N_UNFREEZE = 12)
pour permettre une adaptation plus profonde aux images rétiniennes.


In [32]:
import gc, os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
ACCUM_STEPS = 16
N_UNFREEZE  = 12

# Charger meilleur checkpoint Phase 1
model_rf.load_state_dict(torch.load(f'{MODELS_DIR}/RETFound_phase1.pth', map_location=DEVICE))

# Dégeler les 12 derniers blocs + fc_norm + head
for p in model_rf.parameters():
    p.requires_grad = False
for blk in model_rf.blocks[-N_UNFREEZE:]:
    for p in blk.parameters():
        p.requires_grad = True
for p in model_rf.fc_norm.parameters():
    p.requires_grad = True
for p in model_rf.head.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model_rf.parameters() if p.requires_grad)
print(f"Paramètres entraînables Phase 2 : {trainable/1e6:.1f}M")

backbone_params = [p for blk in model_rf.blocks[-N_UNFREEZE:] for p in blk.parameters()]
head_params     = list(model_rf.fc_norm.parameters()) + list(model_rf.head.parameters())

optimizer_p2 = AdamW([
    {'params': backbone_params, 'lr': 1e-5},
    {'params': head_params,     'lr': 1e-4},
], weight_decay=0.05)
scheduler_p2 = CosineAnnealingLR(optimizer_p2, T_max=20)
scaler       = GradScaler()

best_auc_p2  = 0.0
patience_p2  = 0
MAX_PATIENCE_P2 = 10

print("\n--- Phase 2 : fine-tuning sélectif ---")
for epoch in range(1, 21):
    model_rf.train()
    total_loss = 0
    optimizer_p2.zero_grad()

    for step, (imgs, labels) in enumerate(train_loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        with autocast():
            out  = model_rf(imgs)
            loss = focal_loss_multilabel(out, labels, weights=class_weights)
            loss = loss / ACCUM_STEPS

        scaler.scale(loss).backward()
        total_loss += loss.item() * ACCUM_STEPS

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer_p2)
            torch.nn.utils.clip_grad_norm_(model_rf.parameters(), max_norm=1.0)
            scaler.step(optimizer_p2)
            scaler.update()
            optimizer_p2.zero_grad()

    # Flush des gradients résiduels (si len(loader) n'est pas multiple de ACCUM_STEPS)
    if (step + 1) % ACCUM_STEPS != 0:
        scaler.unscale_(optimizer_p2)
        torch.nn.utils.clip_grad_norm_(model_rf.parameters(), max_norm=1.0)
        scaler.step(optimizer_p2)
        scaler.update()
        optimizer_p2.zero_grad()

    scheduler_p2.step()
    print(f"\nEpoch {epoch} | Loss: {total_loss/len(train_loader):.4f}")
    mean_auc, aucs, _, _ = evaluate_multilabel(model_rf, val_loader)

    if mean_auc > best_auc_p2:
        best_auc_p2 = mean_auc
        torch.save(model_rf.state_dict(), f'{MODELS_DIR}/RETFound_best.pth')
        print(f"  ✓ Sauvegardé (Mean AUC={best_auc_p2:.4f})")
        patience_p2 = 0
    else:
        patience_p2 += 1
        if patience_p2 >= MAX_PATIENCE_P2:
            print(f"  Early stopping Phase 2 à epoch {epoch}")
            break

    gc.collect()
    torch.cuda.empty_cache()

print(f"\nPhase 2 terminée — Best Mean AUC = {best_auc_p2:.4f}")


Paramètres entraînables Phase 2 : 151.2M

--- Phase 2 : fine-tuning sélectif ---

Epoch 1 | Loss: 0.6310
  RD AUC : 0.8804
  Glaucome AUC : 0.9087
  DMLA AUC : 0.8636
  Mean AUC : 0.8842
  ✓ Sauvegardé (Mean AUC=0.8842)

Epoch 2 | Loss: 0.6141
  RD AUC : 0.8807
  Glaucome AUC : 0.9105
  DMLA AUC : 0.8788
  Mean AUC : 0.8900
  ✓ Sauvegardé (Mean AUC=0.8900)

Epoch 3 | Loss: 0.5987
  RD AUC : 0.8820
  Glaucome AUC : 0.9092
  DMLA AUC : 0.8851
  Mean AUC : 0.8921
  ✓ Sauvegardé (Mean AUC=0.8921)

Epoch 4 | Loss: 0.5839
  RD AUC : 0.8838
  Glaucome AUC : 0.9084
  DMLA AUC : 0.8862
  Mean AUC : 0.8928
  ✓ Sauvegardé (Mean AUC=0.8928)

Epoch 5 | Loss: 0.5724
  RD AUC : 0.8850
  Glaucome AUC : 0.9125
  DMLA AUC : 0.8894
  Mean AUC : 0.8956
  ✓ Sauvegardé (Mean AUC=0.8956)

Epoch 6 | Loss: 0.5549
  RD AUC : 0.8859
  Glaucome AUC : 0.9107
  DMLA AUC : 0.8897
  Mean AUC : 0.8954

Epoch 7 | Loss: 0.5505
  RD AUC : 0.8875
  Glaucome AUC : 0.9120
  DMLA AUC : 0.8950
  Mean AUC : 0.8982
  ✓ Sauvegar

### Vérification des Modèles Sauvegardés

Liste des fichiers modèles présents dans `/kaggle/working/models/`
avec leur taille, pour confirmer que tous les entraînements ont bien produit un checkpoint.


In [ ]:
import os
models_dir = '/kaggle/working/models/'
if os.path.exists(models_dir):
    for f in sorted(os.listdir(models_dir)):
        size = os.path.getsize(os.path.join(models_dir, f)) / 1e6
        print(f"{f} : {size:.1f} MB")
else:
    print("Dossier models introuvable")


---
## M5 — Évaluation Finale sur le Test Set

### Métriques calculées par modèle et par pathologie

- **AUC** (Area Under ROC Curve) — métrique principale, indépendante du seuil
- **Sensibilité** et **Spécificité** — au seuil optimal
- **Matrice de confusion** — par pathologie (RD / Glaucome / DMLA)
- **Mean AUC** — moyenne des 3 AUC comme score global du modèle


In [ ]:
from sklearn.metrics import (roc_auc_score, confusion_matrix,
                              classification_report, roc_curve)

def evaluate_full(model_name, preds, labels):
    print(f"\n{'='*50}")
    print(f"Évaluation : {model_name}")
    print(f"{'='*50}")
    results = {}
    for i, disease in enumerate(DISEASES):
        y_true = labels[:, i]
        y_pred = preds[:, i]
        y_bin  = (y_pred >= 0.5).astype(int)

        if y_true.sum() == 0:
            print(f"{disease} : pas d'exemples positifs dans le test")
            continue

        auc  = roc_auc_score(y_true, y_pred)
        cm   = confusion_matrix(y_true, y_bin)
        tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        f1   = 2*tp / (2*tp + fp + fn) if (2*tp + fp + fn) > 0 else 0

        print(f"\n{disease}:")
        print(f"  AUC         : {auc:.4f}")
        print(f"  Sensibilité : {sens:.4f}")
        print(f"  Spécificité : {spec:.4f}")
        print(f"  F1-Score    : {f1:.4f}")

        results[disease] = {'AUC': auc, 'Sensibilité': sens,
                            'Spécificité': spec, 'F1': f1}
    mean_auc = np.mean([v['AUC'] for v in results.values()])
    print(f"\n  Mean AUC : {mean_auc:.4f}")
    results['Mean'] = {'AUC': mean_auc}
    return results

# Évaluation RETFound sur test set
model_rf.load_state_dict(torch.load(f'{MODELS_DIR}/RETFound_best.pth', map_location=DEVICE))
_, _, preds_rf, labels_test = evaluate_multilabel(model_rf, test_loader)
results_rf = evaluate_full('RETFound', preds_rf, labels_test)

### Chargement et Évaluation de RETFound sur le Test Set

Chargement du meilleur checkpoint RETFound (`RETFound_best.pth`) et
évaluation complète sur le test set indépendant.


In [ ]:
import os, cv2, types, torch, timm, numpy as np, pandas as pd
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, confusion_matrix

# Constantes
DISEASES    = ['RD', 'Glaucome', 'DMLA']
IMG_SIZE    = 224
BATCH_SIZE  = 16
MODELS_DIR  = '/kaggle/working/models'
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# CLAHE
def preprocess_clahe(img_bgr):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    channels = cv2.split(img_rgb)
    channels = [clahe.apply(c) for c in channels]
    return cv2.merge(channels)

# Dataset
from albumentations.pytorch import ToTensorV2
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class FundusDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['image_path'])
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        img = preprocess_clahe(img)
        if self.transform:
            img = self.transform(image=img)['image']
        labels = torch.tensor([row['RD'], row['Glaucome'], row['DMLA']], dtype=torch.float32)
        return img, labels

df_test   = pd.read_csv('/kaggle/working/test.csv')
test_loader = DataLoader(FundusDataset(df_test, val_transform),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Recréer model_rf
def _forward_multilabel(self, x):
    x = x.float()
    B = x.shape[0]
    x = self.patch_embed(x)
    cls_tok = self.cls_token.expand(B, -1, -1)
    x = torch.cat((cls_tok, x), dim=1)
    x = x + self.pos_embed
    x = self.pos_drop(x)
    for blk in self.blocks:
        x = blk(x)
    x = x[:, 1:].mean(dim=1)
    x = self.fc_norm(x)
    return self.head(x)

model_rf = timm.create_model('vit_large_patch16_224', pretrained=False,
                              num_classes=3, global_pool='avg')
model_rf.forward = types.MethodType(_forward_multilabel, model_rf)
model_rf.load_state_dict(torch.load(f'{MODELS_DIR}/RETFound_best.pth', map_location=DEVICE))
model_rf = model_rf.to(DEVICE)
model_rf.eval()

# Évaluation
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        probs = torch.sigmoid(model_rf(imgs)).cpu().numpy()
        all_preds.append(probs)
        all_labels.append(labels.numpy())

preds  = np.concatenate(all_preds)
labels = np.concatenate(all_labels)

print("=" * 50)
print("Évaluation RETFound — Test set")
print("=" * 50)
results_rf = {}
for i, disease in enumerate(DISEASES):
    auc  = roc_auc_score(labels[:, i], preds[:, i])
    y_bin = (preds[:, i] >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels[:, i], y_bin, labels=[0,1]).ravel()
    sens = tp / (tp + fn + 1e-8)
    spec = tn / (tn + fp + 1e-8)
    f1   = 2*tp / (2*tp + fp + fn + 1e-8)
    print(f"{disease}: AUC={auc:.4f} | Sens={sens:.4f} | Spec={spec:.4f} | F1={f1:.4f}")
    results_rf[disease] = {'AUC': auc, 'Sensibilité': sens, 'Spécificité': spec, 'F1': f1}

mean_auc = np.mean([results_rf[d]['AUC'] for d in DISEASES])
print(f"Mean AUC = {mean_auc:.4f}")


### Optimisation des Seuils de Décision par Pathologie

Les seuils de décision sont calibrés sur le **val set** (jamais le test set)
pour trouver le meilleur équilibre sensibilité / spécificité par maladie.

Le seuil du **Glaucome est plus bas** car cette pathologie est rare,
le modèle est volontairement plus conservateur pour ne pas manquer de cas.


In [34]:
# CELLULE 15opt — Optimisation seuils par maladie (val set)
from sklearn.metrics import f1_score

# Récupérer prédictions sur val set
model_rf.load_state_dict(torch.load(f'{MODELS_DIR}/RETFound_best.pth', map_location=DEVICE))
_, _, preds_val_rf, labels_val_rf = evaluate_multilabel(model_rf, val_loader)

# Trouver meilleur seuil par maladie
best_thresholds = {}
print("Optimisation des seuils :")
for i, disease in enumerate(DISEASES):
    y_true = labels_val_rf[:, i]
    y_pred = preds_val_rf[:, i]
    best_t, best_f1 = 0.5, 0
    for t in np.arange(0.05, 0.95, 0.05):
        f1 = f1_score(y_true, (y_pred >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    best_thresholds[disease] = best_t
    print(f"  {disease}: seuil optimal = {best_t:.2f} (F1={best_f1:.4f})")

# Réévaluer sur test set avec seuils optimaux
print(f"\n{'='*50}")
print("RETFound — Test set avec seuils optimisés")
print(f"{'='*50}")
results_rf_opt = {}
for i, disease in enumerate(DISEASES):
    y_true = labels_test[:, i]
    y_pred = preds_rf[:, i]
    t      = best_thresholds[disease]
    auc    = roc_auc_score(y_true, y_pred)
    y_bin  = (y_pred >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_bin, labels=[0,1]).ravel()
    sens = tp / (tp + fn + 1e-8)
    spec = tn / (tn + fp + 1e-8)
    f1   = 2*tp / (2*tp + fp + fn + 1e-8)
    print(f"{disease} (seuil={t:.2f}): AUC={auc:.4f} | Sens={sens:.4f} | Spec={spec:.4f} | F1={f1:.4f}")
    results_rf_opt[disease] = {'AUC': auc, 'Sensibilité': sens, 'Spécificité': spec, 'F1': f1}

mean_auc = np.mean([results_rf_opt[d]['AUC'] for d in DISEASES])
results_rf_opt['Mean'] = {'AUC': mean_auc}
print(f"Mean AUC = {mean_auc:.4f}")


  RD AUC : 0.8922
  Glaucome AUC : 0.9150
  DMLA AUC : 0.9037
  Mean AUC : 0.9036
Optimisation des seuils :
  RD: seuil optimal = 0.40 (F1=0.6937)
  Glaucome: seuil optimal = 0.35 (F1=0.4747)
  DMLA: seuil optimal = 0.45 (F1=0.5455)

RETFound — Test set avec seuils optimisés
RD (seuil=0.40): AUC=0.8884 | Sens=0.8467 | Spec=0.7356 | F1=0.6984
Glaucome (seuil=0.35): AUC=0.9226 | Sens=0.6538 | Spec=0.9188 | F1=0.4304
DMLA (seuil=0.45): AUC=0.9185 | Sens=0.4500 | Spec=0.9878 | F1=0.4696
Mean AUC = 0.9098


### Courbes ROC — Comparaison des 4 Modèles par Pathologie

Visualisation des courbes ROC pour RD, Glaucome et DMLA.
L'**AUC moyen** (Mean AUC) est la métrique de comparaison globale entre modèles.


In [ ]:
# CELLULE 16 — Comparaison AUC + Courbes ROC RETFound

from sklearn.metrics import roc_curve, roc_auc_score

COLORS = {
    'DenseNet121':     '#E86B1F',
    'EfficientNetV2S': '#C0392B',
    'EfficientNetB3':  '#8E44AD',
    'RETFound':        '#276FBF',
}

# AUC moyens connus pour tous les modèles
mean_aucs = {
    'DenseNet121':     results_densenet['Mean']['AUC'],
    'EfficientNetV2S': results_effnetv2['Mean']['AUC'],
    'EfficientNetB3':  results_effnet['Mean']['AUC'],
    'RETFound':        np.mean([results_rf[d]['AUC'] for d in DISEASES]),
}

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

# --- Bar chart comparaison ---
ax0 = axes[0]
names = list(mean_aucs.keys())
vals  = list(mean_aucs.values())
bars  = ax0.bar(names, vals, color=[COLORS[n] for n in names], alpha=0.85)
ax0.set_ylim(0.75, 1.0)
ax0.set_ylabel('Mean AUC')
ax0.set_title('Comparaison Mean AUC\n(4 modèles)', fontweight='bold')
ax0.tick_params(axis='x', rotation=15)
for bar, val in zip(bars, vals):
    ax0.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax0.grid(axis='y', alpha=0.3)

# --- Courbes ROC RETFound par maladie ---
y_true_all = np.array(df_test[DISEASES].values, dtype=np.float32)

for i, (disease, ax) in enumerate(zip(DISEASES, axes[1:])):
    y_true = y_true_all[:, i]
    p      = preds_rf[:len(y_true), i]
    fpr, tpr, _ = roc_curve(y_true, p)
    auc_val = roc_auc_score(y_true, p)
    ax.plot(fpr, tpr, color=COLORS['RETFound'], linewidth=2.5,
            label=f'RETFound (AUC={auc_val:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
    ax.set_xlabel('1 - Spécificité (FPR)')
    ax.set_ylabel('Sensibilité (TPR)')
    ax.set_title(f'ROC — {disease}', fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.suptitle('Comparaison des modèles & Courbes ROC RETFound',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'comparaison_auc_roc.png'), dpi=150)
plt.show()
print("Graphiques sauvegardés ✓")


---
## Explicabilité — Grad-CAM

**Gradient-weighted Class Activation Mapping** produit une carte de chaleur
indiquant les régions de l'image les plus influentes pour la décision du modèle.

Cette technique est essentielle en imagerie médicale pour :
- Valider que le modèle cible les bonnes structures anatomiques (macula, disque optique)
- Détecter d'éventuels artefacts capturés comme signal discriminant
- Fournir une explication interprétable aux cliniciens


In [ ]:
import torch.nn.functional as F

class GradCAM:
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, img_tensor, class_idx):
        self.model.eval()
        img_tensor = img_tensor.unsqueeze(0).to(DEVICE)
        img_tensor.requires_grad = True
        output = self.model(img_tensor)
        self.model.zero_grad()
        output[0, class_idx].backward()
        gradients   = self.gradients.mean(dim=[2, 3], keepdim=True)
        activations = self.activations
        cam = (gradients * activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

# Exemple Grad-CAM sur une image de test positive par maladie
def show_gradcam_examples(model, df_test, disease_idx, disease_name, gradcam):
    positive_cases = df_test[df_test[disease_name] == 1].head(3)
    fig, axes = plt.subplots(len(positive_cases), 3, figsize=(12, 4*len(positive_cases)))

    for row_idx, (_, row) in enumerate(positive_cases.iterrows()):
        img_bgr  = cv2.imread(row['image_path'])
        img_rgb  = preprocess_clahe(img_bgr)
        img_orig = cv2.resize(img_rgb, (224, 224))

        tensor = val_transform(image=img_rgb)['image']
        cam    = gradcam.generate(tensor, disease_idx)

        heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        overlay = np.uint8(0.5 * img_orig + 0.5 * heatmap)

        if len(positive_cases) == 1:
            axes[0].imshow(img_orig); axes[0].set_title('Original'); axes[0].axis('off')
            axes[1].imshow(heatmap);  axes[1].set_title('Grad-CAM'); axes[1].axis('off')
            axes[2].imshow(overlay);  axes[2].set_title('Overlay');  axes[2].axis('off')
        else:
            axes[row_idx, 0].imshow(img_orig); axes[row_idx, 0].set_title('Original'); axes[row_idx, 0].axis('off')
            axes[row_idx, 1].imshow(heatmap);  axes[row_idx, 1].set_title('Grad-CAM'); axes[row_idx, 1].axis('off')
            axes[row_idx, 2].imshow(overlay);  axes[row_idx, 2].set_title('Overlay');  axes[row_idx, 2].axis('off')

    plt.suptitle(f'Grad-CAM — {disease_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/gradcam_{disease_name}.png', dpi=150)
    plt.show()

# Pour DenseNet (CNN — Grad-CAM classique sur dernière couche conv)
# Note : pour RETFound (ViT), on utilise l'attention des tokens
# Charger DenseNet entraîné
densenet = tf.keras.models.load_model(f'{MODELS_DIR}/DenseNet121_best.keras')

# Grad-CAM TF pour DenseNet
def gradcam_tf(model, img_array, class_idx):
    last_conv = model.get_layer('conv5_block16_concat')
    grad_model = tf.keras.Model(inputs=model.inputs,
                                outputs=[last_conv.output, model.output])
    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(img_array)
        loss = predictions[:, class_idx]
    grads = tape.gradient(loss, conv_output)[0]
    weights = tf.reduce_mean(grads, axis=(0, 1))
    cam = tf.reduce_sum(weights * conv_output[0], axis=-1)
    cam = tf.maximum(cam, 0) / (tf.math.reduce_max(cam) + 1e-8)
    return cam.numpy()

print("Grad-CAM configuré ✓")
print("Appeler show_gradcam_examples() pour visualiser")

---
## Résultats Finaux — Comparaison des 4 Modèles × 3 Pathologies

### Tableau comparatif (AUC par pathologie)

Comparaison directe des 4 modèles sur le test set indépendant,
pour chacune des 3 pathologies et en AUC moyen global.


In [ ]:
# CELLULE 18 — Tableau comparatif final (4 modèles × 3 maladies)

all_results = {
    'DenseNet121':     results_densenet,
    'EfficientNetV2S': results_effnetv2,
    'EfficientNetB3':  results_effnet,
    'RETFound':        results_rf_opt,
}


rows_table = []
for model_name, res in all_results.items():
    for disease in DISEASES:
        if disease in res:
            rows_table.append({
                'Modèle':      model_name,
                'Maladie':     disease,
                'AUC':         f"{res[disease]['AUC']:.4f}",
                'Sensibilité': f"{res[disease]['Sensibilité']:.4f}",
                'Spécificité': f"{res[disease]['Spécificité']:.4f}",
                'F1':          f"{res[disease]['F1']:.4f}",
            })

df_results = pd.DataFrame(rows_table)
print("\nTableau comparatif :")
print(df_results.to_string(index=False))
df_results.to_csv(os.path.join(RESULTS_DIR, 'resultats_phase2.csv'), index=False)
print("\nRésultats sauvegardés ✓")


### Heatmap AUC — Vue Synthétique

Représentation matricielle des AUC (modèles × pathologies)
pour identifier d'un coup d'œil les forces et faiblesses de chaque architecture.


In [ ]:
# CELLULE 18b — Heatmap AUC finale (4 modèles × 3 maladies)

models_order   = ['DenseNet121', 'EfficientNetV2S', 'EfficientNetB3', 'RETFound']
diseases_order = ['RD', 'Glaucome', 'DMLA']

auc_matrix = np.zeros((len(models_order), len(diseases_order)))
for i, model_name in enumerate(models_order):
    if model_name in all_results:
        for j, disease in enumerate(diseases_order):
            if disease in all_results[model_name]:
                auc_matrix[i, j] = float(all_results[model_name][disease]['AUC'])

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(auc_matrix, cmap='Blues', vmin=0.5, vmax=1.0)
plt.colorbar(im, ax=ax, label='AUC')

ax.set_xticks(range(len(diseases_order))); ax.set_xticklabels(diseases_order, fontsize=12)
ax.set_yticks(range(len(models_order)));   ax.set_yticklabels(models_order,   fontsize=12)
ax.set_title('AUC par modèle et par maladie', fontsize=14, fontweight='bold', pad=15)

for i in range(len(models_order)):
    for j in range(len(diseases_order)):
        val = auc_matrix[i, j]
        color = 'white' if val > 0.75 else 'black'
        ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                fontsize=12, fontweight='bold', color=color)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'heatmap_auc.png'), dpi=150)
plt.show()
print("Heatmap sauvegardée ✓")
print("\n=== PIPELINE PHASE 2 TERMINÉ ===")
print(f"Résultats dans : {RESULTS_DIR}")


---

## Conclusion & Bilan

### Ce qui a été fait

Ce notebook couvre l'intégralité du pipeline : fusion de 6 datasets publics
(~15 600 images de fond d'œil), prétraitement CLAHE + augmentation, split stratifié
anti-fuite de données par patient, puis entraînement et comparaison de **4 architectures**
sur la tâche multi-label RD / Glaucome / DMLA :

| Modèle | Stratégie d'entraînement | Mean AUC (test) |
|---|---|---|
| DenseNet121 | Weighted BCE, 20 epochs | **0.9082** |
| EfficientNetV2S | Weighted BCE, 20 epochs | **0.9179** |
| EfficientNetB3 | Weighted BCE, 20 epochs | **0.9209** |
| RETFound (ViT-Large) | Focal loss, warm-up puis fine-tuning 12 blocs | **0.9098** |

> **Note sur EfficientNetB3** : c'est le modèle retenu pour le déploiement
> (`EfficientNetB3_best.keras`, utilisé par l'application **TriRetina**), avec
> RD = 0.9186, Glaucome = 0.9266, DMLA = 0.9175, **Mean AUC = 0.9209**.
> Ces valeurs sont celles du checkpoint final déployé (voir la note
> méthodologique plus bas pour le détail de la session d'entraînement).

Malgré ses 307M de paramètres et son pré-entraînement sur 1,6M d'images rétiniennes,
RETFound ne surpasse pas les CNN convolutifs sur ce jeu de données de taille modeste,
un résultat cohérent avec la littérature : les gains des ViT pré-entraînés en imagerie
médicale se manifestent surtout à plus grande échelle de données. **EfficientNetB3**
offre le meilleur compromis performance / coût de calcul / taille pour un déploiement
en application (12M de paramètres contre 307M pour RETFound).

### Limites

- **Déséquilibre de classes marqué** : Glaucome (5.9 %) et DMLA (3.2 %) restent
  sous-représentés malgré la pondération de la loss la sensibilité sur ces deux
  pathologies est la métrique la plus fragile (F1 les plus bas du tableau comparatif).
- **Labels hétérogènes entre sources** : certains datasets (REFUGE) sont labellisés
  par convention de nom de fichier plutôt que par grille clinique validée.
- **Seuils optimisés sur le val set** de taille limitée  à recalibrer si le modèle
  est ré-entraîné sur un dataset différent.
- Ce pipeline reste un **outil de recherche / aide au dépistage**, non un dispositif
  médical certifié (cf. avertissements repris dans l'application TriRetina).


### Perspectives 
Le modèle EfficientNetB3 retenu est intégré dans **TriRetina**, l'application
Streamlit développée pour la plateforme de télé-ophtalmologie Temeoo : upload
d'une image de fond d'œil, prédiction des 3 pathologies avec seuils ajustables,
et génération d'un rapport PDF structuré pour l'examinateur.